# 09 — Consolidated Results & Reproduction Verification

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **09 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/— (new terminal step)`

## Objective

Collect every gate written by notebooks 01–08, join them into one summary table, re-verify each gate against the original `mark 1/mark_*_outputs/` JSONs (drift < 1e-4), and render the pipeline status strip.

## Inputs (read-only)

- All 8 gate JSONs from `Evaluation/mark_1_to_4e_outputs/mark_*_outputs/` (written by notebooks 01–08)
- Original gates under `mark 1/mark_*_outputs/` for the reproduction comparison

## Outputs → `Evaluation/output/09_consolidated/data/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `unified_gate_summary.csv` |
| `reproduction_verification.csv` |
| `reproduction_verification.json` |
| `pipeline_status_strip.png` |

**Visualizations produced by this notebook:** `pipeline_status_strip.png`

## Phase dataflow

```mermaid
flowchart LR
  subgraph SRC[shared output root: mark_1_to_4e_outputs]
    direction LR
    G1[mark_1_gate_result.json] & G2[mark_2_gate_result.json]
    G3[mark_3_gate_result.json] & G4[mark_4_gate_result.json]
    G5[mark_4b_gate_result.json] & G6[mark_4c_gate_result.json]
    G7[mark_4d_gate_result.json] & G8[mark_4e_gate_result.json]
  end
  SRC --> C[consolidate + reproduction check vs mark 1 originals]
  C --> O[unified_gate_summary.csv]
  C --> V[reproduction_verification.json]
  C --> S[pipeline status strip]
```


## Key finding (reproduced)

Single source of truth for the whole Mark 1 → 4E research loop: 8 gates, 8 reproduction checks, one `unified_gate_summary.csv` and one status strip.

## Gate

`consolidated/reproduction_verification.json` — every phase must be PASS

## Run notes

**Run last**, after notebooks 01–08. Pure analysis of the shared output folder — no inference, no training, no test access.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "consolidated"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 9 — Consolidated Results & Reproduction Verification

This final section pulls every gate written by Parts 1–8 into one summary table, then **verifies each
recomputed gate against the original `mark 1/` gate JSONs** to prove this unified notebook reproduces
the same outputs and results. A machine-readable report is written to
`Evaluation/output/09_consolidated/data/`.

### 9.1 Consolidated gate summary (Mark 1 → Mark 4E)

In [2]:
GATE_FILES = {
    "mark_1":  ("mark_1_gate_result.json",  "mark_1_outputs"),
    "mark_2":  ("mark_2_gate_result.json",  "mark_2_outputs"),
    "mark_3":  ("mark_3_gate_result.json",  "mark_3_outputs"),
    "mark_4":  ("mark_4_gate_result.json",  "mark_4_outputs"),
    "mark_4b": ("mark_4b_gate_result.json", "mark_4b_outputs"),
    "mark_4c": ("mark_4c_gate_result.json", "mark_4c_outputs"),
    "mark_4d": ("mark_4d_gate_result.json", "mark_4d_outputs"),
    "mark_4e": ("mark_4e_gate_result.json", "mark_4e_outputs"),
}

summary_rows = []
for mark, (fname, sub) in GATE_FILES.items():
    path = load_shared(mark, fname)
    gate = json.loads(path.read_text())
    status = gate.get("status")
    decision = gate.get("decision") or gate.get("next_mark") or gate.get("next_step") or ""
    selected = (gate.get("selected_metrics")
                or gate.get("best_metrics")
                or gate.get("best_observed_configuration_for_diagnosis") or {})
    mean_dice = selected.get("mean_patient_dice", np.nan)
    v104 = selected.get("volume_104_dice", np.nan)
    v116 = selected.get("volume_116_dice", np.nan)
    q1 = selected.get("q1_detected_pct", np.nan)
    pos_empty = selected.get("positive_predicted_empty_pct", np.nan)
    empty_fp = selected.get("empty_slice_false_positive_pct", np.nan)
    summary_rows.append({
        "mark": mark, "status": status, "decision": decision,
        "mean_patient_dice": mean_dice, "volume_104_dice": v104, "volume_116_dice": v116,
        "q1_detected_pct": q1, "positive_predicted_empty_pct": pos_empty,
        "empty_slice_false_positive_pct": empty_fp,
    })

gates_summary = pd.DataFrame(summary_rows)
gates_summary.to_csv(CONSOLIDATED / "unified_gate_summary.csv", index=False)
display(gates_summary)
print(f"Saved consolidated gate summary ({len(gates_summary)} marks).")

,mark,status,decision,mean_patient_dice,volume_104_dice,volume_116_dice,q1_detected_pct,positive_predicted_empty_pct,empty_slice_false_positive_pct
0,mark_1,mark_1_diagnostic_complete,predicted_liver_roi_or_capacity_experiment,0.332869,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027
1,mark_2,mark_2_feasibility_complete,PROCEED_TO_GATED_TWO_STAGE_MULTIWINDOW_OVERFIT,NaN,NaN,NaN,NaN,NaN,NaN
2,mark_3,mark_3_overfit_pass,PROCEED_TO_3_TO_5_EPOCH_TWO_STAGE_VALIDATION_S...,NaN,NaN,NaN,NaN,NaN,NaN
3,mark_4,mark_4_smoke_fail,STOP_AND_DIAGNOSE_TWO_STAGE_SMOKE_FAILURE,0.363866,6.718718e-02,1.078028e-02,42.585551,36.852207,3.422172
4,mark_4b,mark_4b_diagnostic_complete,PROBABILITIES_REMAIN_INSUFFICIENT_REVISE_INPUT...,0.364550,6.455658e-02,1.042448e-02,42.585551,37.044146,3.401431
5,mark_4c,mark_4c_ablation_fail,revise_sampling_or_architecture_before_more_tr...,NaN,NaN,NaN,NaN,NaN,NaN
6,mark_4d,mark_4d_diagnostic_complete_no_full_pass,V116_LOCALIZATION_FAILURE_RUN_SMALL_LESION_SAM...,0.376588,1.002396e-01,7.116977e-04,47.908745,30.518234,4.791040
7,mark_4e,mark_4e_fusion_pass,FREEZE_FUSION_POLICY_AND_THRESHOLD,0.377087,1.166270e-01,1.047355e-02,50.570342,27.447217,5.548066


Saved consolidated gate summary (8 marks).


### 9.2 Reproduction verification vs the original `mark 1/` gates

In [3]:
from pathlib import Path as _P

VERIFY_FIELDS = {
    "mark_1":  ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_2":  ["volume_104_tumor_containment", "volume_116_tumor_containment",
                "minimum_positive_patient_containment", "minimum_positive_slice_containment",
                "median_crop_area_ratio"],
    "mark_3":  ["best_hard_micro_dice", "final_hard_micro_dice", "positive_predicted_empty_pct"],
    "mark_4":  ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4b": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4c": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4d": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4e": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
}

SELECTOR = {
    "mark_1":  lambda g: g["best_observed_configuration_for_diagnosis"],
    "mark_2":  lambda g: g["selected_roi_configuration"],
    "mark_3":  lambda g: g["selected_configuration"],
    "mark_4":  lambda g: g["best_metrics"],
    "mark_4b": lambda g: g["selected_metrics"],
    "mark_4c": lambda g: g["arms"],
    "mark_4d": lambda g: g["selected_metrics"],
    "mark_4e": lambda g: g["selected_metrics"],
}


def compare_gate(mark, recomputed_path, original_path):
    rec = json.loads(_P(recomputed_path).read_text())
    orig = json.loads(_P(original_path).read_text())
    rec_sel = SELECTOR[mark](rec)
    orig_sel = SELECTOR[mark](orig)
    if isinstance(rec_sel, list):  # mark_4c arms
        rec_by_arm = {a["arm"]: a for a in rec_sel}
        orig_by_arm = {a["arm"]: a for a in orig_sel}
        rows = []
        for arm in rec_by_arm:
            for f in VERIFY_FIELDS[mark]:
                diff = abs(float(rec_by_arm[arm][f]) - float(orig_by_arm[arm][f]))
                rows.append({"mark": mark, "selector": arm, "field": f,
                             "recomputed": rec_by_arm[arm][f], "original": orig_by_arm[arm][f],
                             "abs_diff": diff, "passed": diff < 1e-4})
        return rows
    rows = []
    for f in VERIFY_FIELDS[mark]:
        diff = abs(float(rec_sel[f]) - float(orig_sel[f]))
        rows.append({"mark": mark, "selector": mark, "field": f,
                     "recomputed": rec_sel[f], "original": orig_sel[f],
                     "abs_diff": diff, "passed": diff < 1e-4})
    return rows


all_rows = []
for mark, (fname, sub) in GATE_FILES.items():
    recomputed_path = load_shared(mark, fname)
    original_path = MARK1_DIR / sub / fname
    assert original_path.is_file(), f"Missing original gate: {original_path}"
    all_rows.extend(compare_gate(mark, recomputed_path, original_path))

verify_frame = pd.DataFrame(all_rows)
verify_frame.to_csv(CONSOLIDATED / "reproduction_verification.csv", index=False)
passed_count = int(verify_frame["passed"].sum())
total_count = len(verify_frame)
print(f"Reproduction verification: {passed_count}/{total_count} metric comparisons passed.")
display(verify_frame)

report = {
    "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "marks_verified": sorted(verify_frame["mark"].unique().tolist()),
    "comparisons_total": total_count,
    "comparisons_passed": passed_count,
    "all_passed": passed_count == total_count,
    "worst_abs_diff": float(verify_frame["abs_diff"].max()),
    "failures": verify_frame.loc[~verify_frame["passed"]].to_dict(orient="records"),
}
(CONSOLIDATED / "reproduction_verification.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
assert report["all_passed"], "Reproduction verification FAILED — inspect failures above."
print("PASS: unified notebook reproduces every original gate result.")

Reproduction verification: 56/56 metric comparisons passed.


,mark,selector,field,recomputed,original,abs_diff,passed
0,mark_1,mark_1,mean_patient_dice,3.328691e-01,3.328691e-01,0.0,True
1,mark_1,mark_1,volume_104_dice,1.443085e-11,1.443085e-11,0.0,True
2,mark_1,mark_1,volume_116_dice,6.526180e-12,6.526180e-12,0.0,True
3,mark_1,mark_1,q1_detected_pct,2.737643e+01,2.737643e+01,0.0,True
4,mark_1,mark_1,positive_predicted_empty_pct,4.664107e+01,4.664107e+01,0.0,True
5,mark_1,mark_1,empty_slice_false_positive_pct,3.194027e+00,3.194027e+00,0.0,True
6,mark_2,mark_2,volume_104_tumor_containment,1.000000e+00,1.000000e+00,0.0,True
7,mark_2,mark_2,volume_116_tumor_containment,1.000000e+00,1.000000e+00,0.0,True
8,mark_2,mark_2,minimum_positive_patient_containment,1.000000e+00,1.000000e+00,0.0,True
9,mark_2,mark_2,minimum_positive_slice_containment,1.000000e+00,1.000000e+00,0.0,True


{
  "generated_at": "2026-08-11T06:42:22.621598+00:00",
  "marks_verified": [
    "mark_1",
    "mark_2",
    "mark_3",
    "mark_4",
    "mark_4b",
    "mark_4c",
    "mark_4d",
    "mark_4e"
  ],
  "comparisons_total": 56,
  "comparisons_passed": 56,
  "all_passed": true,
  "worst_abs_diff": 0.0,
  "failures": []
}
PASS: unified notebook reproduces every original gate result.


### 9.3 Consolidated findings and interpretation contract

In [4]:
import matplotlib.patches as mpatches

# Pipeline status strip: one tile per mark
figure, axis = plt.subplots(figsize=(18, 3))
axis.set_xlim(0, 16); axis.set_ylim(0, 3); axis.axis("off")
marks = [
    ("Mark 1", "diagnostic", "FAIL (calibration)"),
    ("Mark 2", "ROI feasibility", "PASS"),
    ("Mark 3", "overfit gate", "PASS"),
    ("Mark 4", "smoke test", "5/6 targets"),
    ("Mark 4B", "calibration", "5/6 targets"),
    ("Mark 4C", "ablation", "no arm passed"),
    ("Mark 4D", "reconciliation", "V116 failure"),
    ("Mark 4E", "fusion", "PASS (all 6)"),
]
colors = ["#D66", "#6D6", "#6D6", "#DD6", "#DD6", "#D66", "#D66", "#6D6"]
for i, ((name, what, verdict), color) in enumerate(zip(marks, colors)):
    x = i * 2
    axis.add_patch(plt.Rectangle((x, 0.4), 1.85, 2.0, facecolor=color,
                                 edgecolor="#333", linewidth=1.2))
    axis.text(x + 0.925, 2.05, name, ha="center", va="center", fontsize=11, weight="bold")
    axis.text(x + 0.925, 1.45, what, ha="center", va="center", fontsize=8)
    axis.text(x + 0.925, 0.85, verdict, ha="center", va="center", fontsize=9)
    if i < len(marks) - 1:
        axis.annotate("", xy=(x + 1.9, 1.4), xytext=(x + 1.85, 1.4),
                      arrowprops={"arrowstyle": "->", "linewidth": 1.4})
axis.text(8.0, 2.6, "Mark 1 → Mark 4E research pipeline — unified reproduction",
          ha="center", fontsize=15, weight="bold")
figure.tight_layout()
figure.savefig(CONSOLIDATED_FIGS / "pipeline_status_strip.png", dpi=170, bbox_inches="tight")
plt.show()

print('''
Consolidated interpretation
---------------------------
1. Mark 1 diagnosed suppressed/mis-localized tumor probability (calibration cannot recover it).
2. Mark 2 proved a prediction-only liver ROI contains 100% of every validation tumor (42.7% median crop).
3. Mark 3 proved the frozen ROI geometry + broad window can overfit (Dice 0.9006, 1-channel).
4. Mark 4 smoke reached 5/6 continuation targets; the positive predicted-empty rate stayed high.
5. Mark 4B proved threshold calibration alone cannot fix the recall failure.
6. Mark 4C showed two-channel input collapses and recall-loss alone damages V116.
7. Mark 4D reconciled metrics over the same nine positive patients and isolated V116 as a
   localization failure (median truth-region probability ~ 0), not ROI clipping.
8. Mark 4E: pixelwise maximum fusion of control + recall-loss at threshold 0.70 passes all six
   temporary validation targets -> the controlling inference policy.

Boundaries
----------
- All gates are VALIDATION gates. The test split remains locked until the complete inference policy
  is frozen and the final validation gate passes.
- Temporary continuation targets do not replace final validation targets.
- Threshold selection, fusion weights and checkpoint pairs must stay   frozen before test access.
''')


Consolidated interpretation
---------------------------
1. Mark 1 diagnosed suppressed/mis-localized tumor probability (calibration cannot recover it).
2. Mark 2 proved a prediction-only liver ROI contains 100% of every validation tumor (42.7% median crop).
3. Mark 3 proved the frozen ROI geometry + broad window can overfit (Dice 0.9006, 1-channel).
4. Mark 4 smoke reached 5/6 continuation targets; the positive predicted-empty rate stayed high.
5. Mark 4B proved threshold calibration alone cannot fix the recall failure.
6. Mark 4C showed two-channel input collapses and recall-loss alone damages V116.
7. Mark 4D reconciled metrics over the same nine positive patients and isolated V116 as a
   localization failure (median truth-region probability ~ 0), not ROI clipping.
8. Mark 4E: pixelwise maximum fusion of control + recall-loss at threshold 0.70 passes all six
   temporary validation targets -> the controlling inference policy.

Boundaries
----------
- All gates are VALIDATION gates.

C:\Users\alanm\AppData\Local\Temp\ipykernel_768\955172651.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final takeaways

- **One notebook, eight marks.** The full Mark 1 → Mark 4E research chain now runs as a single
  structured notebook and writes every artifact (gate JSON, CSVs, figures) into
  `Evaluation/mark_1_to_4e_outputs/` with the same naming as the original `mark 1/` outputs.
- **Reproduction verified.** Part 9 compares every recomputed gate against the original gate JSONs
  and asserts all metrics agree to <1e-4 — the unified pipeline is a faithful consolidation.
- **Controlling policy.** `P_fused = max(P_control, P_recall_loss)` at threshold 0.70
  (mean positive-patient Dice 0.3771, Q1 detection 50.57%, positive empty 27.45%, empty FP 5.55%).
- **Next step** (documented in the frozen policy): bounded confirmation of the fused policy, final
  inference-policy freeze, then the one-time locked test evaluation.

In [5]:
# ---- Cross-phase progress tracker (new centralized visualization) ----
cross = pd.read_csv(CONSOLIDATED / "unified_gate_summary.csv")
metric_cols = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
               "q1_detected_pct", "positive_predicted_empty_pct",
               "empty_slice_false_positive_pct"]
figure, axes = plt.subplots(2, 1, figsize=(15, 9))
x = np.arange(len(cross))
width = 0.12
for j, col in enumerate(metric_cols):
    axes[0].bar(x + (j - 2.5) * width, cross[col].fillna(0).to_numpy(), width, label=col)
axes[0].set_xticks(x)
axes[0].set_xticklabels(cross["mark"], rotation=30, ha="right")
axes[0].set_title("Core metrics across Mark 1 -> Mark 4E", fontsize=12, weight="bold")
axes[0].legend(fontsize=8, ncol=2)
axes[0].set_ylabel("value")
axes[1].axis("off")
axes[1].set_title("Gate status / decision strip", fontsize=12, weight="bold")
for j, row in cross.iterrows():
    axes[1].text(0.03, 0.95 - j * 0.11,
                 f"{row['mark']}: {row['status']} - {row['decision']}",
                 fontsize=9, family="monospace", transform=axes[1].transAxes)
figure.tight_layout(rect=(0, 0, 1, 0.97))
save_figure(figure, "cross_phase_progress.png", phase="consolidated")
plt.show()


C:\Users\alanm\AppData\Local\Temp\ipykernel_768\3565685584.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
